In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

result_path = Path("../results/onnx_cpu_fp32.jsonl")

display(df.head())


In [ ]:
cols = [
    "dataset",
    "batch_size",
    "max_length",
    "avg_tokens_per_item",
    "tokenize_tokens_per_sec",
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
    "tokenize_items_per_sec",
    "embedding_items_per_sec",
    "end_to_end_items_per_sec",
    "tokenize_latency_ms_p95",
    "embedding_latency_ms_p95",
    "end_to_end_latency_ms_p95",
]

display(df[cols].sort_values(["dataset", "max_length", "batch_size"]))


In [ ]:
plot_df = df.copy()
plot_df["run"] = (
    plot_df["dataset"].astype(str)
    + ", bs=" + plot_df["batch_size"].astype(str)
    + ", len=" + plot_df["max_length"].astype(str)
)

metrics = [
    "tokenize_tokens_per_sec",
    "embedding_tokens_per_sec",
    "end_to_end_tokens_per_sec",
]

ax = plot_df.plot.bar(
    x="run",
    y=metrics,
    figsize=(16, 6),
)

ax.set_title("BGE-M3 ONNX CPU FP32: Tokenization vs Embedding Throughput")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("tokens/sec")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
latency_metrics = [
    "tokenize_latency_ms_p95",
    "embedding_latency_ms_p95",
    "end_to_end_latency_ms_p95",
]

ax = plot_df.plot.bar(
    x="run",
    y=latency_metrics,
    figsize=(16, 6),
)

ax.set_title("BGE-M3 ONNX CPU FP32: p95 Latency Breakdown")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("p95 latency ms")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
df["tokenization_overhead_ratio"] = (
    df["tokenize_latency_ms_p95"]
) / df["end_to_end_latency_ms_p95"]

plot_df = df.copy()
plot_df["run"] = (
    plot_df["dataset"].astype(str)
    + ", bs=" + plot_df["batch_size"].astype(str)
    + ", len=" + plot_df["max_length"].astype(str)
)

ax = plot_df.plot.bar(
    x="run",
    y="tokenization_overhead_ratio",
    figsize=(16, 5),
    legend=False,
)

ax.set_title("BGE-M3 ONNX CPU FP32: Tokenization Overhead Ratio")
ax.set_xlabel("Benchmark run")
ax.set_ylabel("ratio")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
summary = (
    df.groupby(["dataset", "max_length", "batch_size"], as_index=False)
      .agg(
          tokenize_tokens_per_sec=("tokenize_tokens_per_sec", "median"),
          embedding_tokens_per_sec=("embedding_tokens_per_sec", "median"),
          end_to_end_tokens_per_sec=("end_to_end_tokens_per_sec", "median"),
          end_to_end_latency_ms_p95=("end_to_end_latency_ms_p95", "median"),
      )
)

display(summary)


In [ ]:
for dataset in sorted(summary["dataset"].unique()):
    subset = summary[summary["dataset"] == dataset]

    pivot = subset.pivot_table(
        index="batch_size",
        columns="max_length",
        values="embedding_tokens_per_sec",
    )

    ax = pivot.plot(figsize=(10, 5), marker="o")
    ax.set_title(f"BGE-M3 ONNX CPU FP32: Embedding tokens/sec - dataset={dataset}")
    ax.set_xlabel("batch_size")
    ax.set_ylabel("embedding_tokens_per_sec")
    plt.tight_layout()
    plt.show()
